In [3]:
#import libraries
import cv2
import numpy as np
import pandas as pd
from pyproj import Transformer
from pathlib import Path

In [4]:
# Select Platform
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
#Project Path
if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/Final_Thesis")

else:
    PROJECT_ROOT = Path.cwd()
print("Running in Colab:", IN_COLAB)
print("Project root:", PROJECT_ROOT)

Running in Colab: True
Project root: /content/drive/MyDrive/Final_Thesis


In [6]:
CSV_PATH = PROJECT_ROOT /"lkpg_bus/outputs/runs/track_balsta/yolo12s_bbox_centers_xyxy_lkpg_0.7.csv"
OUTPUT_PATH = PROJECT_ROOT /"lkpg_bus/outputs/output_dynamic_homography.csv"

In [ ]:
FPS = 29.97

HOLD_LAST_H_AFTER_LAST_REFERENCE = True
TRIM_AFTER_LAST_REFERENCE = False

In [ ]:
world_points = np.array([
    [15.623348582379068, 58.41721570910765],
    [15.62350750326398,  58.41729753588714],
    [15.623977186648762, 58.41688310562573],
    [15.62413543697468,  58.41696528436812]
], dtype=np.float64)

In [ ]:
reference_points_by_time = {
    0: np.array([
        [1193.52, 1869.93],
        [809.33, 1578.50],
        [2689.09, 1042.71],
        [2282.73, 862.48]
    ], dtype=np.float32),

    10: np.array([
        [1360.87, 1867.14],
        [975.55, 1567.60],
        [2901.36, 1048.35],
        [2487.60, 862.07]
    ], dtype=np.float32),

    20: np.array([
        [1327.39, 1863.10],
        [940.07, 1563.15],
        [2871.61, 1032.84],
        [2455.86, 847.94]
    ], dtype=np.float32),

    30: np.array([
        [1237.73, 1850.13],
        [847.08, 1555.54],
        [2759.17, 1008.50],
        [2348.40, 827.61]
    ], dtype=np.float32),

    40: np.array([
        [1147.13, 1859.76],
        [753.73, 1567.37],
        [2659.11, 997.65],
        [2247.91, 820.90]
    ], dtype=np.float32),

    50: np.array([
        [1073.23, 1831.34],
        [673.40, 1547.75],
        [2562.36, 954.13],
        [2151.78, 784.38]
    ], dtype=np.float32),

    60: np.array([
        [1001.70, 1826.00],
        [593.49, 1553.40],
        [2449.60, 924.26],
        [2039.74, 763.05]
    ], dtype=np.float32),

    70: np.array([
        [956.43, 1849.01],
        [543.36, 1583.23],
        [2357.32, 926.37],
        [1948.31, 770.90]
    ], dtype=np.float32),

    90: np.array([
        [938.90, 1811.22],
        [519.98, 1544.17],
        [2363.40, 883.57],
        [1955.50, 725.89]
    ], dtype=np.float32),

    100: np.array([
        [769.58, 1815.87],
        [350.28, 1554.47],
        [2172.55, 861.90],
        [1767.54, 713.83]
    ], dtype=np.float32),

    110: np.array([
        [1184.70, 1889.23],
        [840.35, 1566.33],
        [2928.79, 1096.36],
        [2513.71, 886.41]
    ], dtype=np.float32),

    120: np.array([
        [994.50, 1887.02],
        [631.64, 1573.07],
        [2656.33, 1035.79],
        [2251.47, 842.58]
    ], dtype=np.float32),

    130: np.array([
        [963.64, 1879.82],
        [599.77, 1570.92],
        [2618.18, 1026.38],
        [2214.29, 835.00]
    ], dtype=np.float32),

    140: np.array([
        [892.64, 1884.44],
        [525.58, 1578.54],
        [2533.27, 1010.65],
        [2131.81, 826.40]
    ], dtype=np.float32),

    150: np.array([
        [1058.09, 1872.06],
        [697.67, 1556.33],
        [2742.71, 1039.04],
        [2334.64, 843.81]
    ], dtype=np.float32),

    160: np.array([
        [1121.31, 1880.38],
        [766.11, 1564.18],
        [2811.87, 1062.59],
        [2401.21, 862.39]
    ], dtype=np.float32),

    170: np.array([
        [1177.99, 1885.77],
        [824.18, 1562.90],
        [2897.72, 1079.39],
        [2486.18, 873.79]
    ], dtype=np.float32),
}


In [ ]:
def interpolate_image_points(t, reference_dict):
    times = sorted(reference_dict.keys())

    if t <= times[0]:
        return reference_dict[times[0]]

    if t >= times[-1]:
        return reference_dict[times[-1]]

    for i in range(len(times) - 1):
        t0, t1 = times[i], times[i + 1]

        if t0 <= t <= t1:
            P0 = reference_dict[t0]
            P1 = reference_dict[t1]

            alpha = (t - t0) / (t1 - t0)
            Pt = (1 - alpha) * P0 + alpha * P1

            return Pt.astype(np.float32)

    raise ValueError(f"Could not interpolate points for time {t}")

In [ ]:
df = pd.read_csv(CSV_PATH)

required_cols = {"frame_idx", "time_sec", "x1", "y1", "x2", "y2"}
missing = required_cols - set(df.columns)

if missing:
    raise ValueError(f"Missing required columns in CSV: {missing}")

df["x_center_px"] = (df["x1"] + df["x2"]) / 2.0
df["y_center_px"] = (df["y1"] + df["y2"]) / 2.0

last_reference_time = max(reference_points_by_time.keys())

if TRIM_AFTER_LAST_REFERENCE:
    df = df[df["time_sec"] <= last_reference_time].copy()

In [ ]:
max_needed_time = int(np.floor(df["time_sec"].max()))
homographies = {}

for t in range(max_needed_time + 1):

    if t > last_reference_time and not HOLD_LAST_H_AFTER_LAST_REFERENCE:
        continue

    image_points_t = interpolate_image_points(t, reference_points_by_time)

    H_t, _ = cv2.findHomography(
        image_points_t.astype(np.float32),
        world_points.astype(np.float32),
        method=0
    )

    if H_t is None:
        raise ValueError(f"Homography failed at t={t}")

    homographies[t] = H_t

In [ ]:
longitudes = []
latitudes = []

for _, row in df.iterrows():

    t_int = int(np.floor(row["time_sec"]))

    if t_int > last_reference_time:
        if HOLD_LAST_H_AFTER_LAST_REFERENCE:
            t_int = last_reference_time
        else:
            longitudes.append(np.nan)
            latitudes.append(np.nan)
            continue

    H_t = homographies[t_int]

    pixel = np.array(
        [[[row["x_center_px"], row["y_center_px"]]]],
        dtype=np.float32
    )

    lonlat = cv2.perspectiveTransform(pixel, H_t)

    lon = float(lonlat[0, 0, 0])
    lat = float(lonlat[0, 0, 1])

    longitudes.append(lon)
    latitudes.append(lat)

df["longitude"] = longitudes
df["latitude"] = latitudes

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)

print("Done.")
print("Saved:", OUTPUT_PATH)
print(df[[
    "frame_idx",
    "time_sec",
    "x_center_px",
    "y_center_px",
    "longitude",
    "latitude"
]].head())

Done.
Saved: /content/drive/MyDrive/Final_Thesis/lkpg_bus/outputs/output_dynamic_homography.csv
   frame_idx  time_sec  x_center_px  y_center_px  longitude   latitude
0          1  0.033367  1735.182983   858.051086  15.624117  58.417103
1          1  0.033367  2448.043213  1300.248413  15.623754  58.416958
2          1  0.033367  3159.094482   940.031555  15.624094  58.416756
3          1  0.033367   305.008392  1932.453735  15.623291  58.417374
4          1  0.033367  1166.105194  1119.077881  15.623854  58.417236
